# From GRPO to DAPO: Four Fixes for Long-Reasoning RL

This notebook develops DAPO as one coordinated improvement over GRPO. Each section follows the same rhythm: **predict → inspect → optimize → explain**.

The four DAPO techniques are:

1. **Clip-Higher** — preserve exploration
2. **Dynamic Sampling** — select informative prompt groups
3. **Token-Level Policy Gradient Loss** — balance token contributions
4. **Overlong Reward Shaping** — soften the truncation boundary

> Current status: Section 1 is implemented. Sections 2–4 will be added to this same notebook.


## 1. Clip-Higher: Give Rare Successful Tokens Room to Grow

### Learning question

> A rare reasoning strategy succeeds. Why can ordinary GRPO struggle to increase its probability?

Imagine a geometry problem where the model chooses its opening strategy:

| Opening token | Old probability | Interpretation |
|---|---:|---|
| `Let` | 0.80 | Familiar algebraic approach |
| `Draw` | 0.02 | Rare diagram-based approach |
| Other tokens | 0.18 | Other approaches |

A sampled response beginning with `Draw` solves the problem and receives positive advantage. GRPO measures how its probability changes using the importance ratio

$$r_t = \frac{\pi_\theta(\text{Draw}\mid s_t)}{\pi_{\text{old}}(\text{Draw}\mid s_t)}.$$


### Predict

The old probability of `Draw` is 0.02. Before running the next cell, predict its probability when the ratio first reaches:

- GRPO's symmetric upper bound, $1 + 0.2 = 1.2$
- DAPO's higher upper bound, $1 + 0.5 = 1.5$


In [ ]:
p_old = 0.02
eps_symmetric = 0.20
eps_high = 0.50

p_at_symmetric_ceiling = p_old * (1 + eps_symmetric)
p_at_clip_higher_ceiling = p_old * (1 + eps_high)

grpo_room = p_at_symmetric_ceiling - p_old
dapo_room = p_at_clip_higher_ceiling - p_old

print(f"Old probability:          {p_old:.3f}")
print(f"GRPO clipping threshold:  {p_at_symmetric_ceiling:.3f}")
print(f"DAPO clipping threshold:  {p_at_clip_higher_ceiling:.3f}")
print(f"\nRoom supplied by Clip-Higher: {dapo_room / grpo_room:.1f}x as much")


For this sampled token-context pair, GRPO's positive-advantage surrogate becomes flat after the ratio reaches 1.2. DAPO keeps rewarding the probability increase until the ratio reaches 1.5.

The threshold is **not a permanent hard maximum** on the token's probability: other samples and later rollout batches can still change it. It marks where this sample stops supplying a positive gradient.


### Inspect the clipped objective

For positive advantage $A>0$, the clipped surrogate is

$$L(r,A)=\min\left(rA,\operatorname{clip}(r,1-\epsilon_{\mathrm{low}},1+\epsilon_{\mathrm{high}})A\right).$$

The next plot shows exactly where each objective stops increasing.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ratios = np.linspace(0.5, 1.8, 400)
advantage = 1.0
eps_low = 0.2

def clipped_surrogate(ratios, advantage, eps_low, eps_high):
    unclipped = ratios * advantage
    clipped = np.clip(
        ratios,
        1 - eps_low,
        1 + eps_high,
    ) * advantage
    return np.minimum(unclipped, clipped)

grpo_objective = clipped_surrogate(
    ratios, advantage, eps_low=0.2, eps_high=0.2
)
dapo_objective = clipped_surrogate(
    ratios, advantage, eps_low=0.2, eps_high=0.5
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ratios, grpo_objective, label="GRPO: upper bound 1.2", linewidth=3)
ax.plot(ratios, dapo_objective, label="DAPO: upper bound 1.5", linewidth=3)
ax.axvline(1.2, color="C0", linestyle="--", alpha=0.6)
ax.axvline(1.5, color="C1", linestyle="--", alpha=0.6)
ax.set(
    xlabel="Probability ratio: new / old",
    ylabel="Objective for positive advantage",
    title="Clip-Higher lets a successful token improve further",
)
ax.legend()
ax.grid(alpha=0.2)
plt.show()


### Optimize the same rollout more than once

Clipping matters only after the current policy moves away from the policy that generated the rollout. We therefore freeze the old probability and take multiple optimization steps on the same positive-advantage sample.

This one-action Bernoulli model is a local microscope, not a complete language model. It isolates the clipping mechanism without hiding it inside a neural network.


In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def logit(p):
    return np.log(p / (1 - p))

def optimize_positive_token(
    p_old,
    eps_high,
    learning_rate=0.01,
    epochs=250,
):
    """Optimize one positive-advantage action against a frozen rollout policy."""
    z = logit(p_old)
    history = []

    for epoch in range(epochs):
        p_new = sigmoid(z)
        ratio = p_new / p_old
        gradient_active = ratio < 1 + eps_high

        # For A=1, d(ratio)/dz = ratio * (1 - p_new).
        gradient = ratio * (1 - p_new) if gradient_active else 0.0

        history.append({
            "epoch": epoch,
            "probability": p_new,
            "ratio": ratio,
            "gradient_active": gradient_active,
        })
        z += learning_rate * gradient

    return history

grpo_history = optimize_positive_token(p_old=0.02, eps_high=0.20)
dapo_history = optimize_positive_token(p_old=0.02, eps_high=0.50)

print(f"GRPO final probability: {grpo_history[-1]['probability']:.4f}")
print(f"DAPO final probability: {dapo_history[-1]['probability']:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    [x["epoch"] for x in grpo_history],
    [x["probability"] for x in grpo_history],
    label="GRPO: symmetric clipping",
    linewidth=3,
)
ax.plot(
    [x["epoch"] for x in dapo_history],
    [x["probability"] for x in dapo_history],
    label="DAPO: Clip-Higher",
    linewidth=3,
)
ax.axhline(p_old * 1.2, color="C0", linestyle="--", alpha=0.6)
ax.axhline(p_old * 1.5, color="C1", linestyle="--", alpha=0.6)
ax.set(
    xlabel="Optimization epoch on the same rollout",
    ylabel='Probability of rare successful token "Draw"',
    title="Where does the positive-advantage update stop?",
)
ax.legend()
ax.grid(alpha=0.2)
plt.show()


### Explain

> GRPO's upper clipping bound can stop a rare successful token after only a small absolute probability increase. DAPO raises the upper bound, allowing exploration tokens to become meaningfully more likely before their positive update is clipped.

Clip-Higher does **not** add an entropy bonus, guarantee that entropy always rises, or remove clipping. It gives positively advantaged tokens more room to grow. DAPO reports that tokens hitting the upper clip tend to be low-probability exploration tokens, so widening the upper range helps preserve diversity.


### Modify: learner exercise

Try old probabilities `[0.50, 0.10, 0.02, 0.001]` and upper clipping values `[0.20, 0.30, 0.50, 0.80]`. Plot the allowed absolute increase $p_{\mathrm{old}}\epsilon_{\mathrm{high}}$.

Discuss:

1. Which tokens receive the smallest absolute increase?
2. Why do repeated rollout-and-update cycles matter?
3. Why might an extremely large upper bound make training less conservative?


In [ ]:
old_probabilities = np.array([0.50, 0.10, 0.02, 0.001])
upper_clip_values = np.array([0.20, 0.30, 0.50, 0.80])

# TODO: Create one curve per old probability.
# Hint: allowed absolute increase = p_old * eps_high


---

## 2. Dynamic Sampling

Next: compare easy, learnable, and impossibly hard prompt groups, then measure useful gradients per generated sample.

## 3. Token-Level Policy Gradient Loss

Next: use genuinely variable-length responses to compare sample-level and token-level aggregation.

## 4. Overlong Reward Shaping

Next: add an explicit end token and compare hard truncation with a soft length penalty.

## Reference

Yu et al. (2025), [DAPO: An Open-Source LLM Reinforcement Learning System at Scale](https://arxiv.org/abs/2503.14476).
